In [2]:
print(123)


123


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from openai import OpenAI
openai_client = OpenAI()

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s going on?'

In [7]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes — if the course is still open, you can usually join now.

A few things to check:
- **Enrollment deadline:** Some courses allow late enrollment, some don’t.
- **Prerequisites:** Make sure you meet any required background or access conditions.
- **Catch-up options:** Ask whether recordings, notes, or starter materials are available.
- **Current status:** If the course has already started, see whether you can join the next session or be added to the current one.

If you want, I can help you draft a short message to the instructor or course admin asking if late entry is possible.


In [8]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [9]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [10]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still open.


## Getting the data

In [11]:
import requests
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(courses_raw)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 85}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

len(documents)

1350

In [13]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

## Building Search Engine

In [14]:
from minsearch import Index

index = Index(
    text_fields = ['quesiton', 'section', 'answer'], #all fields that are useful for search
    keyword_fields = ['course']
)

index.fit(documents)

In [15]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form 

In [16]:
[doc["question"] for doc in search_results]

['Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'Homework: Why does the content keep changing?',
 'How should I start the course and follow the weekly workflow?',
 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?']

In [17]:
[doc['section'] for doc in search_results]

['General Course-Related Questions',
 'General Course-Related Questions',
 'General Course-Related Questions',
 'General Course-Related Questions',
 'General Course-Related Questions']

In [18]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [19]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

## Building a prompt

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
INSTRUCTIONS = '''Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.'''

In [ ]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}'''

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context)
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)

In [ ]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

General Course-Related Questions
Q: Homework: Why does the content keep 

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text



In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [ ]:
response.output[0]

ResponseOutputMessage(id='msg_0433d571e01da128006a36dce04104819ebf1431186d3eac51', content=[ResponseOutputText(annotations=[], text='Yes — you can join now.\n\nYou can start learning and submitting homework as long as the homework form is open. You don’t need a confirmation email or a registered list to begin.\n\nA couple of notes:\n- If you want a certificate, that’s only available if you complete the course with a live cohort.\n- If the homework is marked `[DRAFT]`, it may still change.\n\nIf you want, I can also point you to the best place to start today.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [ ]:
response.output[0].content[0].text

'Yes — you can join now.\n\nYou can start learning and submitting homework as long as the homework form is open. You don’t need a confirmation email or a registered list to begin.\n\nA couple of notes:\n- If you want a certificate, that’s only available if you complete the course with a live cohort.\n- If the homework is marked `[DRAFT]`, it may still change.\n\nIf you want, I can also point you to the best place to start today.'

In [ ]:
response.usage

ResponseUsage(input_tokens=605, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=705)

In [ ]:
input_price = 0.76 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0009098

In [ ]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [ ]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
answer = rag("What are the requirements for the final project?")
print(answer)

The final project requirements are:

- It is an **individual project**, not a group project.
- Your project must be **independently evaluable**.
- It should be **reproducible**: a reviewer must be able to clone your repo and follow the README to recreate it from scratch.
- **Do not include API keys or hosted-service credentials** in the repo.
- Provide a **script or notebook** that ingests the data and rebuilds the search index locally.
- Include a **`.env.example`** with variable names only; the reviewer should use their own `.env`.
- Use a **cheap model** so reviewers do not spend too many credits.
- **Pin dependency versions** and document the **Python version** and, if applicable, the **Docker version**.

If you want, I can also summarize the **grading/evaluation rules** for the capstone.


In [23]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

AttributeError: 'Response' object has no attribute 'output'